# Implementasi Bab 3 — Missing Value, Outlier, dan Imbalanced Data

Implementasi KNNImputer, IsolationForest, SMOTE, undersampling, dan class weight pada data sintetis tidak seimbang [ML-MODUL-01, hlm. 26–27]. Adaptasi penting: split dilakukan sebelum fitting imputasi/outlier/SMOTE agar test set tidak bocor.


In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.datasets import make_classification
from sklearn.ensemble import IsolationForest
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
X_array, y = make_classification(
    n_samples=2000,
    n_features=8,
    n_informative=5,
    weights=[0.95, 0.05],
    random_state=SEED,
)
feature_names = [f"fitur_{index}" for index in range(8)]
X = pd.DataFrame(X_array, columns=feature_names)
rng = np.random.default_rng(SEED)
X = X.mask(rng.random(X.shape) < 0.05)
print("Missing per fitur:\n", X.isna().sum())


In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

imputer = KNNImputer(n_neighbors=5)
X_train = pd.DataFrame(
    imputer.fit_transform(X_train_raw), columns=feature_names, index=X_train_raw.index
)
X_test = pd.DataFrame(
    imputer.transform(X_test_raw), columns=feature_names, index=X_test_raw.index
)

detector = IsolationForest(contamination=0.02, random_state=SEED)
inlier_mask = detector.fit_predict(X_train) == 1
X_train_clean = X_train.loc[inlier_mask]
y_train_clean = pd.Series(y_train, index=X_train_raw.index).loc[inlier_mask]
print("Outlier train dibuang:", int((~inlier_mask).sum()))


In [ ]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train_clean, y_train_clean)

smote = SMOTE(random_state=SEED)
X_train_smote, y_train_smote = smote.fit_resample(X_train_clean, y_train_clean)
model_smote = LogisticRegression(max_iter=1000, random_state=SEED)
model_smote.fit(X_train_smote, y_train_smote)

for name, candidate in [("Baseline", baseline), ("SMOTE", model_smote)]:
    prediction = candidate.predict(X_test)
    print(name, "accuracy=", round(accuracy_score(y_test, prediction), 3),
          "F1=", round(f1_score(y_test, prediction), 3))


In [ ]:
before = pd.Series(y_train_clean).value_counts().sort_index()
after = pd.Series(y_train_smote).value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].bar(before.index.astype(str), before.values)
axes[0].set_title("Sebelum SMOTE")
axes[1].bar(after.index.astype(str), after.values)
axes[1].set_title("Setelah SMOTE")
plt.show()


## Latihan 1: mean imputation vs KNN imputation

Perbandingan dilakukan dengan fitting imputer hanya pada train set. F1 dipakai karena kelas positif langka; akurasi saja dapat menyesatkan [ML-MODUL-01, hlm. 27].


In [ ]:
imputation_scores = {}
for name, candidate_imputer in {
    "Mean": SimpleImputer(strategy="mean"),
    "KNN": KNNImputer(n_neighbors=5),
}.items():
    train_i = candidate_imputer.fit_transform(X_train_raw)
    test_i = candidate_imputer.transform(X_test_raw)
    candidate = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
    candidate.fit(train_i, y_train)
    imputation_scores[name] = f1_score(y_test, candidate.predict(test_i))
print(imputation_scores)


In [ ]:
sampler = RandomUnderSampler(random_state=SEED)
X_train_under, y_train_under = sampler.fit_resample(X_train_clean, y_train_clean)
strategies = {
    "SMOTE": model_smote,
    "Undersampling": LogisticRegression(max_iter=1000, random_state=SEED).fit(X_train_under, y_train_under),
    "Class weight": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED).fit(X_train_clean, y_train_clean),
}
strategy_scores = {
    name: f1_score(y_test, candidate.predict(X_test))
    for name, candidate in strategies.items()
}
display(pd.Series(strategy_scores, name="F1").sort_values(ascending=False).to_frame())


## Kesimpulan

Strategi terbaik tidak selalu yang memberi akurasi tertinggi. Bandingkan F1/recall, jaga semua transformasi tetap belajar dari train set, dan pilih penanganan imbalance sesuai biaya kesalahan kasus nyata.
